# <u>Convolutional Neural Network (CNN)</u>

### Prerequisites:
* <a href="../Artificial Neural Networks/Neural Networks.ipynb">Check out the notebook on Neural Networks</a>

## Topics

* [0. Introduction](#intro)
* [1. Convolutional Operation](#op)
* [2. Properties of Convolution](#prop)
* [3. CNN Components](#comp)
* [4. CNN Applications](#app)
* [5. 1D / 2D / 3D Convolutions](#1D2D3D)
* [6. Important Types of Convolutions](#important)
* [7. Separable Convolutions and Flattening](#separable)
* [8. Modern Architectures - I](#1)
* [9. Modern Architectures - II](#2)
* [10. CNN Library](#lib)


In [ ]:
import numpy as np # for random numbers and numpy arrays
import pandas as pd # for dataframes
import matplotlib.pyplot as plt # for 2D Plots


print("Setup complete")

Setup complete


In [19]:
image = np.array([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 16]
])

# 2x2 filter
kernel = np.array([
    [2, 0],
    [0, -2]
])

stride = 1

output_size = (image.shape[0] - kernel.shape[0]) // stride + 1
output = np.zeros((output_size, output_size))

for i in range(output_size):
    for j in range(output_size):

        # Extract the current 2x2 region
        region = image[i:i+2, j:j+2]

        # Element-wise multiplication and sum
        output[i, j] = np.sum(region * kernel)

print(output)

[[-10. -10. -10.]
 [-10. -10. -10.]
 [-10. -10. -10.]]


In [ ]:
# Input image
image = np.array([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 16]
])

# 2x2 filter
kernel = np.array([
    [2, 0],
    [0, -2]
])

stride = 1

output_size = (image.shape[0] - kernel.shape[0]) // stride + 1
output_vec = []

for row in range(output_size):
    for col in range(output_size):
        region = image[row:row+2, col:col+2]
        output_vec.append(np.sum(region * kernel))

output = np.array(output_vec).reshape(output_size, output_size)

print(output)

[[-10 -10 -10]
 [-10 -10 -10]
 [-10 -10 -10]]


<a class="anchor" id="intro"></a>

# 0. Introduction

### <u>1. What are CNNs?</u>

- CNNs are a class of deep neural networks inspired by the human visual cortex
- They process data with a spatial structure (such as images) by learning hierarchical features automatically
- Early layers learn simple features (e.g., edges), while deeper layers learn more complex patterns (e.g., shapes and objects)

### <u>2. Common applications</u>

* Image classification – assigning labels to images
* Object detection and localization – identifying and locating objects
* Semantic segmentation – classifying every pixel in an image
* Autonomous driving – mapping camera images directly to steering commands 
* Road segmentation from aerial imagery
* Medical imaging – COVID-19 detection and localization from CT scans
* Computational pathology – segmenting cell nuclei in microscopic images
* Image colorization – generating color versions of grayscale images
* Speech and audio analysis – emotion recognition from speech
- CNNs are also applied beyond images, including natural language processing, audio, and time-series data

### <u>3. Basic CNN architecture</u>
1. Input layer
    - Receives the raw input (image, audio, etc.)
2. Convolutional layers
    - Apply learnable filters (kernels) to extract feature maps
    - These layers detect increasingly complex features as the network gets deeper
3. Pooling layers
    - Reduce the spatial dimensions of the feature maps
    - Help retain important information while reducing computation and overfitting
4. Fully connected layers
    - Combine the extracted features to make a prediction
5. Softmax layer
    - Converts the final outputs into class probabilities for classification tasks

<div style="display: flex; justify-content: center; gap: 10px;">
  <img src="pics/1.png" width="600"/>
</div>


<a class="anchor" id="op"></a>

# 1. Convolutional Operation

### <u>1. Filters extract features</u>

Before CNNs, computer vision relied on hand-crafted filters to detect specific image features. One of the most famous examples is the Sobel filter, which detects image edges.

##### Why edges?

Edges occur where neighboring pixel intensities change rapidly. The Sobel operator approximates the image gradient, measuring how quickly brightness changes across the image.

### <u>2. Sobel filters</u>

Given an image $A$ we apply the filters using the **convolution** $\ast$.

##### Horizontal gradient $G_x=S_x \ast A$
- Detects vertical edges:
$$
S_x = \begin{bmatrix} -1 & 0 & \color{green}1\color{white} \\ -2 & 0 & \color{green}2\color{white} \\ \color{green}-1\color{white} & \color{green}0\color{white} & \color{green}1\color{white} \end{bmatrix}
$$
$\qquad \quad$- $S_x$ is the outer product of $\underbrace{\begin{bmatrix} \color{green}1\color{white} \\ \color{green}2\color{white} \\ \color{green}1\color{white} \end{bmatrix}}_{\text{averaging}} \in \mathbb{R}^{3 \times 1}$ and $\underbrace{\begin{bmatrix} \color{green}-1\color{white} & \color{green}0\color{white} & \color{green}1\color{white} \end{bmatrix}}_{\text{differentiation}} \in \mathbb{R}^{1 \times 3}$

$\qquad \quad$ - The column vector $\begin{bmatrix} 1 \\ 2 \\ 1 \end{bmatrix}$ is an averaging (or smoothing) kernel with the center value weighted more heavily

$\qquad \quad$ - Normalizing it yields $\begin{bmatrix} 1/4 \\ 2/4 \\ 1/4 \end{bmatrix}$ and it computes a weighted average of neighboring pixels

$\qquad \quad$ - Given vertical pixel values like $\begin{bmatrix} 100 \\ 110 \\ 120 \end{bmatrix}$ the weighted average is $\frac{100 + 2 \cdot 110 + 120 }{4}=110$ 

$\qquad \quad$ - So instead of looking at one noisy pixel, consider its neighbors

$\qquad \quad$ - The row vector $\begin{bmatrix} -1 & 0 & 1 \end{bmatrix}$ computes a finite difference

$\qquad \quad$ - For three neighboring pixels $\begin{bmatrix} 50 & 60 & 120 \end{bmatrix}$ it computes $-50+0+120=70$

$\qquad \quad$ - If the left and right pixels are similar, $\begin{bmatrix} 80 & 82 & 84 \end{bmatrix}$ then $-80+84=4$

$\qquad \quad$ - This kernel therefore measures how quickly the intensity changes from left to right

##### Horizontal gradient $G_y=S_y \ast A$
- Detects horizontal edges:
$$
S_y = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ 1 & 2 & 1 \end{bmatrix}
$$


The overall edge strength is computed as 

$$
G=\sqrt{G_x^2 + G_y^2}
$$

which combines gradients from both directions.


### <u>3. Applying a filter</u>

Images are represented as matrices of pixel values.

To apply a filter:
1. Place the filter over a small region of the image.
2. Multiply corresponding entries.
3. Sum the products.
4. Store the result in the output feature map.
5. Move the filter one position and repeat.

Applying the Sobel filter at every location creates a feature map, where high values indicate detected edges. Because the filter cannot be centered on border pixels without padding, the output is smaller than the input.


### <u>4. Connection to CNNs</u>

The key insight is that CNNs perform the same convolution operation, but with an important difference:

- Traditional computer vision uses manually designed filters like Sobel
- CNNs start with randomly initialized filters which are then applied to the input
- During training, gradient descent updates the filter weights so they learn useful features automatically


### <u>5. Images and filters in CNNs</u>

Grey scale images ...
- are matrices of dimensions $\text{Height} \times \text{Width} \times \text{Depth}$ with $\text{Depth}=1$
- have Pixel entries that differ from 0 (black) to 255 (white)

Color images ...
- are tensors of dimensions $\text{Height} \times \text{Width} \times \text{Depth}$ with $\text{Depth}=3$
- depth value denotes the RGB values (red- green- blue)

Filters ...
- are usually square
- depth value must always equal to the input's depth

**For example, a $2 \times 2$ filter applied to an RGB image actually has dimensions $2 \times 2 \times 3$.**


### <u>6. Mathematical definition of 2D convolution</u>

For a $3 \times 3 \times 1$ input 
$$
\begin{bmatrix} a & b & c \\ d & e & f \\ g & h & i \end{bmatrix}
$$

and a $2 \times 2 \times 1$ filter

$$
\begin{bmatrix} w_{11} & w_{12}  \\ w_{21} & w_{22}   \end{bmatrix}
$$

the output is a $2 \times 2 \times 1$ feature map whose entries are computed as dot products:

$$
\begin{align*}
s_{11} &= aw_{11} + bw_{12} + dw_{21} + ew_{22} \\
s_{12} &= bw_{11} + cw_{12} + ew_{21} + fw_{22} \\
s_{21} &= dw_{11} + ew_{12} + gw_{21} + hw_{22} \\
s_{22} &= ew_{11} + fw_{12} + hw_{21} + iw_{22} \\
\end{align*}
$$

<div style="display: flex; justify-content: center; gap: 10px;">
  <img src="pics/2.png" width="550"/>
  <img src="pics/3.png" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 10px;">
  <img src="pics/4.png" width="550"/>
  <img src="pics/5.png" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 10px;">
  <img src="pics/6.png" width="550"/>
  <img src="pics/7.png" width="550"/>
</div>



### <u>7. Key takeaways</u>
- A filter (kernel) is a small matrix that extracts specific patterns from an image.
- Traditional filters like Sobel detect edges using fixed, manually designed weights.
- Convolution slides the filter across the image, computing a dot product at each position to produce a feature map.
- CNNs use the same convolution operation, but their filters are learned automatically during training rather than manually specified.
- For color images, the filter's depth must match the image's depth (e.g., $3 \times 3 \times 3$ for RGB images).

<a class="anchor" id="prop"></a>

# 2. Properties of Convolution

### <u>1. Sparse interactions</u>

Unlike a dense neural network, where every neuron is connected to every neuron in the previous layer, a convolutional layer only connects each output neuron to a small local region of the input (called the receptive field).

**Example:**
For a $3 \times 3 \times 1$ input and a $2 \times 2 \times 1$ filter:
- Each output value depends on only 4 input pixels 
- So our receptive field are 4 neurons and we apply a "local search" for featrures
- The convolution therefore creates 16 total connections

<div style="display: flex; justify-content: center; gap: 10px;">
  <img src="pics/8.png" width="550"/>
  <img src="pics/9.png" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 10px;">
  <img src="pics/10.png" width="550"/>
  <img src="pics/11.png" width="550"/>
</div>


- A comparable fully connected layer would require 36 connections because every hidden neuron connects to all 9 input pixels
- The receptive field of the dense network are 9 neurons
- So we apply a "global search" for features

<div style="display: flex; justify-content: center; gap: 10px;">
  <img src="pics/12.png" width="550"/>
</div>

**Why is this useful?**

Images have strong local structure. For example, when detecting a person's eye, nearby pixels (like the face) are much more informative than pixels on the opposite side of the image. Therefore, CNNs search for features locally, while dense networks unnecessarily consider the entire image.


### <u>2. Parameter sharing</u>

One of the biggest advantages of CNNs is that the same filter weights are reused across the entire image.

Instead of learning a different set of weights for every image location, a CNN slides one filter over the input.

For example, a $2 \times 2 \times 1$ filter has only four weights:
$$
\begin{bmatrix} w_{11} & w_{12} \\ w_{21} & w_{22} \end{bmatrix}
$$
As the filter moves across the image, these same four weights are reused repeatedly.

**Comparison with dense networks**
Using the previous example:

- CNN: 4 learnable weights
<div style="display: flex; justify-content: center; gap: 10px;">
  <img src="pics/13.png" width="550"/>
  <img src="pics/14.png" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 10px;">
  <img src="pics/15.png" width="550"/>
  <img src="pics/16.png" width="550"/>
</div>

---
<div style="display: flex; justify-content: center; gap: 10px;">
  <img src="pics/17.png" width="550"/>
</div>


- Dense network: 36 learnable weights

<div style="display: flex; justify-content: center; gap: 10px;">
  <img src="pics/18.png" width="550"/>
</div>

The dense network therefore uses 9 times more parameters.

### <u>3. Why sparse connections and parameter sharing matter</u>

These two properties provide several advantages:
- Far fewer parameters to learn
- Lower memory usage
- Faster computation
- Reduced risk of overfitting, leading to better generalization
- Dense Layer: For $m$ inputs and $n$ outputs the dense net requires $m \times n$ parameters and runtime
- Convolutional layer: Requires $k \times n$ parameters and runtime, where $k \ll m$ is the much smaller receptive field

**Large-scale example**

For a $100 \times 100 \times 3$ RGB image:
- A CNN using a $5 \times 5 \times 3$ filter needs only $5 \times 5 \times 3 = 75$ parameters

- A comparable dense layer would require $\underbrace{(100^2 \cdot 3)}_{\text{input layer}} \cdot \underbrace{100^2}_{\text{hidden layer}} = 300.000.000$ parameters. 

This illustrates why CNNs scale much better to large images.

### <u>4. Equivariance to translation</u>

A convolutional filter searches for the same feature everywhere in the image.

If a learned feature (such as an eye or an edge) moves to a different location, the filter will still detect it because the same weights are applied at every position.

This property is called equivariance to translation. In simple terms:

- Move the object $\rightarrow$ the detected feature moves by the same amount in the feature map.

The filter does not depend on the absolute position of the feature, only on whether the local pattern matches the learned weights.

### <u>5. Nonlinearity with ReLU</u>

After convolution, CNNs apply an activation function to every value in the feature map.

A common activation function is ReLU (Rectified Linear Unit): $$\text{ReLU}(x)=\max(0,x)$$

ReLU is preferred because it:

- introduces nonlinearity,
- helps reduce the vanishing gradient problem compared to sigmoid,
- produces sparse activations by setting negative values to zero, improves computational efficiency


### <u>6. Key Takeaways</u>

- Sparse interactions: Each output neuron processes only a small neighborhood of the input instead of the whole image.
- Parameter sharing: A single filter is reused across the entire image, dramatically reducing the number of learnable parameters.
- Equivariance to translation: The same learned feature can be detected regardless of where it appears in the image.
- ReLU activations: Add nonlinearity, improve training, and increase computational efficiency.

Together, these properties make CNNs significantly more efficient and effective than fully connected networks for image analysis.


<a class="anchor" id="comp"></a>

# 3. CNN Components


### <u>1. Input channels</u>

Images are represented as matrices of pixel values.



<a class="anchor" id="app"></a>

# 4. CNN Applications

<a class="anchor" id="1D2D3D"></a>

# 5. 1D / 2D / 3D Convolutions

<a class="anchor" id="important"></a>

# 6. Important Types of Convolutions

<a class="anchor" id="separable"></a>

# 7. Separable Convolutions and Flattening

<a class="anchor" id="1"></a>

# 8. Modern Architectures - I

<a class="anchor" id="2"></a>

# 9. Modern Architectures - II

<a class="anchor" id="lib"></a>

# 10. CNN Library